This NB is based on the original rule_cleaning_classification.ipynb and tests the performance of GPT4-mini

In [3]:
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#import importlib
from os.path import join
import logging
import warnings
warnings.filterwarnings('ignore')

import time
from openai import OpenAI
pd.set_option('display.max_colwidth', 255)
from sklearn.metrics import precision_score, recall_score, f1_score
from single_prompt import prompt

from config import OPENAI_API_KEYS

In [16]:
logging.basicConfig(
    filename="process_rules.log",
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

In [ ]:
client = OpenAI(api_key=OPENAI_API_KEYS[0])

In [ ]:
df = pd.read_parquet(join('predictions_gpt', 'rule_manual_annotation_cleaned.parquet'))
df.head()

,Unnamed: 0,name,joined_rules,translated_rules,be_nice,dick,hate_speech,covenant,harassment,CP,...,scrape_model_training,private_no_reg,impersonation,purchase_mbship,block_report_mute,no_fediblock,freedom,anti_capitalist,public_order_morals,no_rule
0,0,toot.re,"plaats geen (links of referenties naar) racistische, hatelijke of offensieve content\n plaats geen content die in nederland illegaal is plaats geen spam gebruik de inhoudswaarschuwing (cw) voor sexueel expliciete content\n geen ruimte voor nepaccounts...","Do not place (left or references to) racist, hateful or offensive content\n Do not place content that is illegal in the Netherlands. Do not use spam use the content warning (CW) for sexually explicit content\n No room for fake accounts.Clear satire of...",NaN,NaN,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,nebu.social,have a friendly demeanor. independence and self-reliance enjoy yours and others hobbies respectful behaviour,have a friendly demeanor. independence and self-reliance enjoy yours and others hobbies respectful behaviour,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,adinfinitum.social,"be nice! hate speech and slurs are not allowed. it is possible to disagree without being disagreeable. be a person. for-profit companies & brands are not permitted. causes and nonprofits are allowed here. this is a welcoming space. no racism, sexism, ...","be nice! hate speech and slurs are not allowed. it is possible to disagree without being disagreeable. be a person. for-profit companies & brands are not permitted. causes and nonprofits are allowed here. this is a welcoming space. no racism, sexism, ...",1.0,NaN,1.0,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,fedi.opensourcestories.org,"be open and welcoming prioritize inclusion act with empathy and kindness respect differing opinions, viewpoints, and experiences no harassment no racism, sexism, ableism, homophobia, transphobia, or other expressions of hate no trolling","be open and welcoming prioritize inclusion act with empathy and kindness respect differing opinions, viewpoints, and experiences no harassment no racism, sexism, ableism, homophobia, transphobia, or other expressions of hate no trolling",1.0,NaN,1.0,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,politiek.social,elke gebruiker is een volksvertegenwoordiger elke gebruiker is bekend onder zijn/haar eigen naam,Each user is a people's representative Each user is known under his/her own name,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# mean length of translated rules per instance
df.translated_rules.map(lambda x: len(x) if type(x)==str else 0).mean()

np.float64(416.4277108433735)

### Category grouping

In [6]:
df = df.rename(columns={'dick': 'dont_be_dick'})

In [57]:
CATEGORIES_INIT = ['be_nice', 'dont_be_dick',
       'hate_speech', 'covenant', 'harassment', 'CP', 'zoophilia', 'NSFW',
       'politics', 'religion', 'porn', 'violence', 'porn_nocw',
       'violent_no_cw', 'nsfw_no_cw', 'spoilers_no_cw', 'sensitive_no_cw',
       'politics_no_cw', 'cw_general', 'no_cw_general',
       'bots_unmarked_no_permission', 'gov_accounts',
       'far_right_extremist_nationalism_fascism', 'illegal',
       'consequences_moderation', 'spam', 'incitement_violence',
       'mis_disinfo_conspiracy_diffamation_propaganda',
       'doxxing_personal_info', 'ads', 'images', 'brigading_dogpiling',
       'behavior_content_format_allowed', 'codes_conduct',
       'copyright_piracy_credit', 'server_interference_disruption',
       'automated_tools_bots', 'topic_specific', 'language_specific',
       'low_quality_content', 'reposting_crossposting', 'trolling',
       'links_outside_content', 'min_age', 'spoilers', 'crypto', 'ai_content',
       'need_use_real_name', 'reverse_systemic_oppression',
       'scrape_model_training', 'private_no_reg', 'impersonation',
       'purchase_mbship', 'block_report_mute', 'no_fediblock', 'freedom',
       'anti_capitalist', 'public_order_morals', 'no_rule']

In [58]:
len(CATEGORIES_INIT)

59

In [59]:
for category in CATEGORIES_INIT:
    if category == 'private_no_reg':
        df[category] = df[category].astype(str).str.strip()
        df[category] = df[category].replace('', np.nan)
        df[category] = df[category].astype(float)
    df[category] = df[category].fillna(0)

In [60]:
def compute_cat_stats(df, categories):
    category_counts = df[categories].sum().sort_values(ascending=False)
    category_percentages = (category_counts / len(df)) * 100
    return category_counts, category_percentages

In [61]:
total_per_cat, relative_per_cat = compute_cat_stats(df, CATEGORIES_INIT)
total_per_cat

hate_speech                                      223.0
be_nice                                          179.0
illegal                                          176.0
harassment                                       135.0
mis_disinfo_conspiracy_diffamation_propaganda    103.0
spam                                             100.0
doxxing_personal_info                             84.0
ads                                               70.0
incitement_violence                               69.0
violence                                          67.0
dont_be_dick                                      67.0
porn_nocw                                         64.0
consequences_moderation                           63.0
porn                                              62.0
violent_no_cw                                     54.0
private_no_reg                                    52.0
far_right_extremist_nationalism_fascism           52.0
brigading_dogpiling                               47.0
nsfw_no_cw

In [62]:
relative_per_cat

hate_speech                                      44.779116
be_nice                                          35.943775
illegal                                          35.341365
harassment                                       27.108434
mis_disinfo_conspiracy_diffamation_propaganda    20.682731
spam                                             20.080321
doxxing_personal_info                            16.867470
ads                                              14.056225
incitement_violence                              13.855422
violence                                         13.453815
dont_be_dick                                     13.453815
porn_nocw                                        12.851406
consequences_moderation                          12.650602
porn                                             12.449799
violent_no_cw                                    10.843373
private_no_reg                                   10.441767
far_right_extremist_nationalism_fascism          10.4417

# Merge and clean cats

In [26]:
def merge_cats(df, cats_to_merge, target_cat):
    """Sum all categories from cats_to_merge to target_cat"""
    for i in range(df.shape[0]):
        for column in cats_to_merge:
            if df[column][i] == 1:
                df[target_cat][i] = 1
    return df.drop(columns=cats_to_merge)

In [27]:
cw_columns_list = ['porn_nocw', 'violent_no_cw', 'nsfw_no_cw', 'spoilers_no_cw', 'sensitive_no_cw', 'politics_no_cw']#, 'no_cw_general']
df = merge_cats(df=df, cats_to_merge=cw_columns_list, target_cat='cw_general')

In [125]:
bots_cats = ['bots_unmarked_no_permission', 'scrape_model_training']
df = merge_cats(df=df, cats_to_merge=bots_cats, target_cat='automated_tools_bots')

In [126]:
violence_cats = ['incitement_violence']
df = merge_cats(df=df, cats_to_merge=violence_cats, target_cat='violence')

In [127]:
troll_cats = ['dont_be_dick']
df = merge_cats(df=df, cats_to_merge=troll_cats, target_cat='trolling')

In [128]:
len([c for c in df.columns if c in CATEGORIES_INIT])

49

In [129]:
total_per_cat, relative_per_cat = compute_cat_stats(df, [c for c in df.columns if c in CATEGORIES_INIT])
total_per_cat

hate_speech                                      223.0
be_nice                                          179.0
illegal                                          176.0
cw_general                                       155.0
harassment                                       136.0
violence                                         115.0
mis_disinfo_conspiracy_diffamation_propaganda    103.0
spam                                             100.0
doxxing_personal_info                             84.0
trolling                                          77.0
ads                                               70.0
consequences_moderation                           63.0
porn                                              62.0
private_no_reg                                    52.0
far_right_extremist_nationalism_fascism           48.0
brigading_dogpiling                               47.0
copyright_piracy_credit                           42.0
CP                                                39.0
automated_

In [130]:
relative_per_cat

hate_speech                                      44.6
be_nice                                          35.8
illegal                                          35.2
cw_general                                       31.0
harassment                                       27.2
violence                                         23.0
mis_disinfo_conspiracy_diffamation_propaganda    20.6
spam                                             20.0
doxxing_personal_info                            16.8
trolling                                         15.4
ads                                              14.0
consequences_moderation                          12.6
porn                                             12.4
private_no_reg                                   10.4
far_right_extremist_nationalism_fascism           9.6
brigading_dogpiling                               9.4
copyright_piracy_credit                           8.4
CP                                                7.8
automated_tools_bots        

In [131]:
categories = [c for c in df.columns if c in CATEGORIES_INIT]
category_counts = df[categories].sum()

categories_with_few_instances = category_counts[category_counts <= 20]
uncommon_categories_list = categories_with_few_instances.index.tolist()

print("Categories covering 20 instances or fewer:")
print(uncommon_categories_list)

Categories covering 20 instances or fewer:
['covenant', 'zoophilia', 'NSFW', 'politics', 'religion', 'no_cw_general', 'gov_accounts', 'images', 'behavior_content_format_allowed', 'server_interference_disruption', 'low_quality_content', 'reposting_crossposting', 'links_outside_content', 'spoilers', 'crypto', 'ai_content', 'need_use_real_name', 'reverse_systemic_oppression', 'purchase_mbship', 'no_fediblock', 'freedom', 'anti_capitalist', 'public_order_morals', 'no_rule']


In [132]:
df = df.drop(columns=uncommon_categories_list)

In [133]:
df.columns

Index(['name', 'joined_rules', 'translated_rules', 'be_nice', 'hate_speech',
       'harassment', 'CP', 'porn', 'violence', 'cw_general',
       'far_right_extremist_nationalism_fascism', 'illegal',
       'consequences_moderation', 'spam',
       'mis_disinfo_conspiracy_diffamation_propaganda',
       'doxxing_personal_info', 'ads', 'brigading_dogpiling', 'codes_conduct',
       'copyright_piracy_credit', 'automated_tools_bots', 'topic_specific',
       'language_specific', 'trolling', 'min_age', 'private_no_reg',
       'impersonation', 'block_report_mute'],
      dtype='object')

In [134]:
len(list(df.columns))

28

In [135]:
df.head()

,name,joined_rules,translated_rules,be_nice,hate_speech,harassment,CP,porn,violence,cw_general,...,codes_conduct,copyright_piracy_credit,automated_tools_bots,topic_specific,language_specific,trolling,min_age,private_no_reg,impersonation,block_report_mute
0,toot.re,"plaats geen (links of referenties naar) racistische, hatelijke of offensieve content\n plaats geen content die in nederland illegaal is plaats geen spam gebruik de inhoudswaarschuwing (cw) voor sexueel expliciete content\n geen ruimte voor nepaccounts...","Do not place (left or references to) racist, hateful or offensive content\n Do not place content that is illegal in the Netherlands. Do not use spam use the content warning (CW) for sexually explicit content\n No room for fake accounts.Clear satire of...",0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,nebu.social,have a friendly demeanor. independence and self-reliance enjoy yours and others hobbies respectful behaviour,have a friendly demeanor. independence and self-reliance enjoy yours and others hobbies respectful behaviour,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,adinfinitum.social,"be nice! hate speech and slurs are not allowed. it is possible to disagree without being disagreeable. be a person. for-profit companies & brands are not permitted. causes and nonprofits are allowed here. this is a welcoming space. no racism, sexism, ...","be nice! hate speech and slurs are not allowed. it is possible to disagree without being disagreeable. be a person. for-profit companies & brands are not permitted. causes and nonprofits are allowed here. this is a welcoming space. no racism, sexism, ...",1.0,1.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3,fedi.opensourcestories.org,"be open and welcoming prioritize inclusion act with empathy and kindness respect differing opinions, viewpoints, and experiences no harassment no racism, sexism, ableism, homophobia, transphobia, or other expressions of hate no trolling","be open and welcoming prioritize inclusion act with empathy and kindness respect differing opinions, viewpoints, and experiences no harassment no racism, sexism, ableism, homophobia, transphobia, or other expressions of hate no trolling",1.0,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,politiek.social,elke gebruiker is een volksvertegenwoordiger elke gebruiker is bekend onder zijn/haar eigen naam,Each user is a people's representative Each user is known under his/her own name,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [136]:
df.loc[df['translated_rules'].isnull()]

,name,joined_rules,translated_rules,be_nice,hate_speech,harassment,CP,porn,violence,cw_general,...,codes_conduct,copyright_piracy_credit,automated_tools_bots,topic_specific,language_specific,trolling,min_age,private_no_reg,impersonation,block_report_mute
25,mordor.social,"*nutzungsbedingungen, datenschutz und datensicherheit* version 1.2, vom 17. märz 2021. betreiberinformationen gemäß § 5 tmg steffen krumbholz olbrichtstraße 6a 04157 leipzig deutschland e-mail: hostmaster(at)libranet(dot)de info: https://mordor.social...",NaN,1.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
470,wirnity.de,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [137]:
df = df.loc[~df['translated_rules'].isnull()].reset_index(drop=True)

In [138]:
df['translated_rules'] = df['translated_rules'].apply(lambda x: x.lower())

In [139]:
final_categories = [x for x in list(df.columns) if x not in ['name', 'joined_rules', 'translated_rules']]

In [140]:
df.head()

,name,joined_rules,translated_rules,be_nice,hate_speech,harassment,CP,porn,violence,cw_general,...,codes_conduct,copyright_piracy_credit,automated_tools_bots,topic_specific,language_specific,trolling,min_age,private_no_reg,impersonation,block_report_mute
0,toot.re,"plaats geen (links of referenties naar) racistische, hatelijke of offensieve content\n plaats geen content die in nederland illegaal is plaats geen spam gebruik de inhoudswaarschuwing (cw) voor sexueel expliciete content\n geen ruimte voor nepaccounts...","do not place (left or references to) racist, hateful or offensive content\n do not place content that is illegal in the netherlands. do not use spam use the content warning (cw) for sexually explicit content\n no room for fake accounts.clear satire of...",0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,nebu.social,have a friendly demeanor. independence and self-reliance enjoy yours and others hobbies respectful behaviour,have a friendly demeanor. independence and self-reliance enjoy yours and others hobbies respectful behaviour,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,adinfinitum.social,"be nice! hate speech and slurs are not allowed. it is possible to disagree without being disagreeable. be a person. for-profit companies & brands are not permitted. causes and nonprofits are allowed here. this is a welcoming space. no racism, sexism, ...","be nice! hate speech and slurs are not allowed. it is possible to disagree without being disagreeable. be a person. for-profit companies & brands are not permitted. causes and nonprofits are allowed here. this is a welcoming space. no racism, sexism, ...",1.0,1.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3,fedi.opensourcestories.org,"be open and welcoming prioritize inclusion act with empathy and kindness respect differing opinions, viewpoints, and experiences no harassment no racism, sexism, ableism, homophobia, transphobia, or other expressions of hate no trolling","be open and welcoming prioritize inclusion act with empathy and kindness respect differing opinions, viewpoints, and experiences no harassment no racism, sexism, ableism, homophobia, transphobia, or other expressions of hate no trolling",1.0,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,politiek.social,elke gebruiker is een volksvertegenwoordiger elke gebruiker is bekend onder zijn/haar eigen naam,each user is a people's representative each user is known under his/her own name,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [141]:
df['private_no_reg'][479] = 0

In [142]:
df['private_no_reg'].value_counts(dropna=False)

private_no_reg
0.0    446
1.0     52
Name: count, dtype: int64

In [143]:
len(final_categories)

25

In [144]:
final_categories

['be_nice',
 'hate_speech',
 'harassment',
 'CP',
 'porn',
 'violence',
 'cw_general',
 'far_right_extremist_nationalism_fascism',
 'illegal',
 'consequences_moderation',
 'spam',
 'mis_disinfo_conspiracy_diffamation_propaganda',
 'doxxing_personal_info',
 'ads',
 'brigading_dogpiling',
 'codes_conduct',
 'copyright_piracy_credit',
 'automated_tools_bots',
 'topic_specific',
 'language_specific',
 'trolling',
 'min_age',
 'private_no_reg',
 'impersonation',
 'block_report_mute']

In [145]:
for category in final_categories:
    df[category] = df[category].astype(int)

# regex

In [146]:
category_words_dict = {'be_nice': ['nice', 'respect'],
                      'hate_speech': ['hate', 'speech', 'racism'],
                      'harassment': ['harass'],
                      'porn': ['sexual', 'porn', 'nudity'],
                      'violence': ['violence'],
                      'cw_general': ['content warning', 'cw', 'mark'],
                      'far_right_extremist_nationalism_fascism': ['extreme', 'nazi', 'fascis', 'nationalis'],
                      'illegal': ['illegal'],
                      'consequences_moderation': ['banned', 'suspended', 'deleted', 'removed', 'moderat'],
                      'spam': ['spam'],
                      'mis_disinfo_conspiracy_diffamation_propaganda': ['false', 'conspiracy', 'misinformation'],
                      'doxxing_personal_info': ['dox', 'personal information'],
                      'ads': ['ads', 'advertis'],
                      'brigading_dogpiling': ['brigading', 'dogpiling'],
                      'codes_conduct': ['code of conduct', 'http'],
                      'copyright_piracy_credit': ['copyright', 'credit'],
                      'automated_tools_bots': ['bot', 'automated', 'scrap'],
                      'topic_specific': ['topic', 'specific'],
                      'language_specific': ['english', 'german', 'japanese'],
                      'min_age': ['age', '18', 'minor'],
                      'private_no_reg': ['private', 'personal', 'test',  'family'],
                      'impersonation': ['identity', 'impersonat'],
                      'block_report_mute': ['block', 'report', 'mute'],
                      'trolling': ['dick', 'jerk', 'troll'],
                      'CP': ['child', 'cp']}

In [ ]:
evaluation_results_regex = pd.DataFrame(columns=['Category', 'Precision_regex', 'Recall_regex', 'F1-score_regex'])

def text_to_label(X, category):
    regex_pattern = ''
    if category in category_words_dict:
        category_words = category_words_dict[category]
        regex_pattern = '|'.join(category_words)
    return X.apply(lambda x: 1 if re.search(regex_pattern, x, flags=re.IGNORECASE) else 0)

for category in final_categories:
    print(category)
    X = df['translated_rules'] 
    y = df[category]

    y_pred = text_to_label(X, category)
    precision = precision_score(y, y_pred)
    recall = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)
#   number_positives = np.sum(y)
    new_row = pd.DataFrame([{'Category': category,
                         'Precision_regex': precision,
                         'Recall_regex': recall,
                         'F1-score_regex': f1}])

    evaluation_results_regex = pd.concat([evaluation_results_regex, new_row], ignore_index=True)

be_nice
hate_speech
harassment
CP
porn
violence
cw_general
far_right_extremist_nationalism_fascism
illegal
consequences_moderation
spam
mis_disinfo_conspiracy_diffamation_propaganda
doxxing_personal_info
ads
brigading_dogpiling
codes_conduct
copyright_piracy_credit
automated_tools_bots
topic_specific
language_specific
trolling
min_age
private_no_reg
impersonation
block_report_mute


In [63]:
evaluation_results_regex.head(50)

,Category,Precision_regex,Recall_regex,F1-score_regex
0,be_nice,0.900990,0.511236,0.652330
1,hate_speech,0.946237,0.792793,0.862745
2,harassment,0.984000,0.904412,0.942529
3,CP,0.880000,0.578947,0.698413
4,porn,0.413043,0.919355,0.570000
5,violence,0.870968,0.710526,0.782609
6,cw_general,0.931973,0.883871,0.907285
7,far_right_extremist_nationalism_fascism,0.719298,0.854167,0.780952
8,illegal,0.969466,0.725714,0.830065
9,consequences_moderation,0.587302,0.596774,0.592000


# OpenAI

## prompt per cat

In [78]:
category_prompt = dict()

In [ ]:
category_prompt['be_nice'] = 'Does this rule encourage users to be nice and respect each other?'
category_prompt['dont_be_dick'] = 'Does this rule ask users to not be a jerk/dick?'
category_prompt['hate_speech'] = 'Does this rule prohibit hate speech, either generally or any subtype of hate speech (e.g., racism)?'
category_prompt['harassment'] = 'Does this rule prohibit harassment?'
category_prompt['porn'] = 'Does this rule prohibit porn?'
category_prompt['violence'] = 'Does this rule prohibit violence, either direct or incitement?'
category_prompt['cw_general'] = 'Does this rule generally encourage or force users to use content warnings for specific content?'
category_prompt['far_right_extremist_nationalism_fascism'] = 'Does this rule consider that supporters of the far right, nazis, fascists, nationalists or other extremists are not welcome on the instance?'
category_prompt['illegal'] = 'Does this rule prohibit illegal content?'
category_prompt['consequences_moderation'] = 'Does this rule indicate the consequences taken if someones behavior breaks the rules?'
category_prompt['spam'] = 'Does this rule prohibit spam?'
category_prompt['mis_disinfo_conspiracy_diffamation_propaganda'] = 'Does this rule prohibit false information, misinformation, disinformation, conspiracy theories, diffamation or propaganda?'
category_prompt['doxxing_personal_info'] = 'Does this rule prohibit doxxing, the sharing of personal information without permission?'
category_prompt['ads'] = 'Does this rule prohibit advertising?'
category_prompt['brigading_dogpiling'] = 'Does this rule prohibit brigading or dogpiling?'
category_prompt['codes_conduct'] = 'Does this rule mention a code of conduct or another kind of community guidelines or document?'
category_prompt['copyright_piracy_credit'] = 'Does this rule force users to respect copyright and give credit where necessary?'
category_prompt['automated_tools_bots'] = 'Does this rule prohibit bots or other automated behavior?'
category_prompt['topic_specific'] = 'Is this instance focused on a specific topic?'
category_prompt['language_specific'] = 'Does this rule expect or recommend users to use a certain language or a certain group of languages?'
category_prompt['min_age'] = 'Does this instance have a minimum age?'
category_prompt['private_no_reg'] = 'Does this rule specify that the instance is a private, personal or single-user instance, a test or development server or invite-only?'
category_prompt['impersonation'] = 'Does this rule prohibit impersonation?'
category_prompt['block_report_mute'] = 'Does this rule encourage users to block, report or mute behavior they do not like?'

In [80]:
final_categories

['be_nice',
 'hate_speech',
 'harassment',
 'CP',
 'porn',
 'violence',
 'cw_general',
 'far_right_extremist_nationalism_fascism',
 'illegal',
 'consequences_moderation',
 'spam',
 'mis_disinfo_conspiracy_diffamation_propaganda',
 'doxxing_personal_info',
 'ads',
 'brigading_dogpiling',
 'codes_conduct',
 'copyright_piracy_credit',
 'automated_tools_bots',
 'topic_specific',
 'language_specific',
 'trolling',
 'min_age',
 'private_no_reg',
 'impersonation',
 'block_report_mute']

## single prompt

In [17]:
# single prompt

def categorize_rule(rule, prompt, max_retries=2, sleep_seconds=30):
    
    fallback = {
        "error": True,
        "error_type": None,
        "raw_response": None,
    }

    if not isinstance(rule, str) or not rule.strip():
        fallback["error_type"] = "empty_rule"
        return fallback

    content = f"{prompt}\n{rule}"


    response = None
    for attempt in range(1, max_retries + 1):
        try:
            response = client.responses.create(
                model="gpt-5-mini",
                input=[{"role": "user", "content": content}],
            )
            break
        except Exception as e:
            logging.error(f"API call failed (attempt {attempt}): {e}")
            if attempt < max_retries:
                time.sleep(sleep_seconds)
            else:
                fallback["error_type"] = "api_call_failed"
                fallback["raw_response"] = str(e)
                return fallback


    text = ""
    try:
        for item in response.output:
            if item.type == "message" and item.content:
                text += item.content[0].text
    except Exception as e:
        fallback["error_type"] = "response_parse_failed"
        fallback["raw_response"] = str(e)
        return fallback

    text = text.strip()
    fallback["raw_response"] = text

    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1 or end <= start:
        fallback["error_type"] = "no_json_found"
        return fallback

    json_text = text[start:end + 1]

    try:
        output_dict = json.loads(json_text)
        return output_dict
    except Exception as e:
        fallback["error_type"] = "json_parse_failed"
        logging.error(f"JSON parse failed: {e}\n{text}")
        return fallback

In [18]:
def process_rules(df_with_rules, start_idx):
    batch_size = 1000
    results = []
    print(f"{pd.Timestamp.now()}: starting process.")
    logging.info(f"Starting process at index {start_idx}")

    for i, (idx, row) in enumerate(df_with_rules.iterrows(), start=start_idx+1):
    
        text = row['translated_joined_rules']
        response = categorize_rule(text, prompt)
    
        results.append({
            "name": row["name"],
            **{f"gpt5-{k}": v for k, v in response.items()}
        })
    
        if i % batch_size == 0:
            start = i - batch_size
            end = i - 1
            filename = f"partial_results_{start:04d}_{end:04d}.parquet"
            pd.DataFrame(results).to_parquet(filename)
            print(f"{pd.Timestamp.now()}: saved {filename}")
            logging.info(f"{pd.Timestamp.now()}: saved {filename}")
            results = []
    
    
    if results:
        start = (i // batch_size) * batch_size
        end = i
        filename = f"partial_results_{start:04d}_{end:04d}.parquet"
        pd.DataFrame(results).to_parquet(filename)
        logging.info(f"{pd.Timestamp.now()}: saved {filename}")
        
    logging.info(f"{pd.Timestamp.now()}: finished process.")


## classify all rules

In [4]:
df_final_rules = pd.read_parquet(join("..", "final_rules_dataset_w_translations.parquet"))

In [5]:
df_final_rules.shape

(15916, 11)

In [9]:
df_with_rules = df_final_rules[df_final_rules['has_rules']==1].copy()

In [10]:
df_with_rules.shape

(8277, 11)

In [ ]:
start_idx = 0
df_resume = df_with_rules.iloc[start_idx:]
process_rules(df_resume, start_idx)

2025-12-17 09:59:21.308454: starting process.


## classify annotated sample 

In [69]:
# gpt-5
df_expanded = df['translated_rules'].apply(
    lambda r: pd.Series({f"gpt5-{k}": v for k, v in categorize_rule(r, prompt).items()})
)
df_full = pd.concat([df, df_expanded], axis=1)
df_full.to_parquet("predictions-rules-gpt_5_mini-single_prompt.parquet", index=False)

In [ ]:
# gpt-5-mini
for cat in final_categories:
    try:
        print(f"{pd.Timestamp.now()}: starting {cat}.")
        prompt = category_prompt[cat]
        result = df[['translated_rules', cat]].copy()
        result[f'gpt_5_mini_{cat}'] = result['translated_rules'].map(lambda x: categorize_rule(x, prompt))
        result.to_parquet(f"predictions-rules-gpt_5_mini_{cat}.parquet", index=False)
        print(f"{pd.Timestamp.now()}: saved {cat}.")
    except Exception as e:
        print(e)
        continue

2025-12-03 13:37:06.532822: starting be_nice.
2025-12-03 14:02:05.958541: saved be_nice.
2025-12-03 14:02:05.958774: starting hate_speech.
2025-12-03 14:31:05.332593: saved hate_speech.
2025-12-03 14:31:05.333267: starting harassment.
2025-12-03 14:56:05.130136: saved harassment.
2025-12-03 14:56:05.131853: starting CP.
'CP'
2025-12-03 14:56:05.131995: starting porn.
2025-12-03 15:23:36.608453: saved porn.
2025-12-03 15:23:36.608791: starting violence.
2025-12-03 15:59:37.978241: saved violence.
2025-12-03 15:59:37.978785: starting cw_general.
2025-12-03 16:26:56.886352: saved cw_general.
2025-12-03 16:26:56.886845: starting far_right_extremist_nationalism_fascism.


### gpt4-mini

In [544]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

In [545]:
dict1 = load_pickle('be_nice-hate_speech-harassment-porn-violence-cw_general.pickle')
dict2 = load_pickle('illegal-consequences_moderation-spam-mis_disinfo-doxxing.pickle')
dict3 = load_pickle('far_right_extremist_nationalism_fascism-ads-brigading_dogpiling-codes_conduct-copyright_piracy_credit-automated_tools_bots-topic_specific-language_specific-min_age-private_no_reg-impersonation-block_report_mute.pickle')

In [546]:
dict1_list = dict1['be_nice']
dict2_list = dict2['illegal']
dict3_list = dict3['far_right_extremist_nationalism_fascism']

In [547]:
def distribute_list(dictx, dictx_list):
    chunk_size = 498
    keys = list(dictx.keys())

    for i, key in enumerate(keys):
        start = i * chunk_size
        end = start + chunk_size
        dictx[key] = dictx_list[start:end]

In [548]:
distribute_list(dict1, dict1_list)
distribute_list(dict2, dict2_list)
distribute_list(dict3, dict3_list)

In [550]:
all_rules_categories = {}

dict_keys([])

In [551]:
all_rules_categories.update(dict1)

In [552]:
all_rules_categories.keys()

dict_keys(['be_nice', 'hate_speech', 'harassment', 'porn', 'violence', 'cw_general'])

In [553]:
all_rules_categories.update(dict2)

In [554]:
all_rules_categories.keys()

dict_keys(['be_nice', 'hate_speech', 'harassment', 'porn', 'violence', 'cw_general', 'far_right_extremist_nationalism_fascism', 'illegal', 'consequences_moderation', 'spam', 'mis_disinfo_conspiracy_diffamation_propaganda', 'doxxing_personal_info'])

In [555]:
all_rules_categories.update(dict3)

In [556]:
all_rules_categories.keys()

dict_keys(['be_nice', 'hate_speech', 'harassment', 'porn', 'violence', 'cw_general', 'far_right_extremist_nationalism_fascism', 'illegal', 'consequences_moderation', 'spam', 'mis_disinfo_conspiracy_diffamation_propaganda', 'doxxing_personal_info', 'ads', 'brigading_dogpiling', 'codes_conduct', 'copyright_piracy_credit', 'automated_tools_bots', 'topic_specific', 'language_specific', 'min_age', 'private_no_reg', 'impersonation', 'block_report_mute'])

In [533]:
len(all_rules_categories.keys())

23

In [534]:
set(final_categories)==set(list(all_rules_categories.keys()))

False

In [535]:
len(final_categories)

25

In [536]:
set(final_categories).difference(set(list(all_rules_categories.keys())))

{'CP', 'trolling'}

In [593]:
columns_to_extract = final_categories + ['name', 'joined_rules', 'translated_rules']
columns_to_extract.remove('CP')
columns_to_extract.remove('trolling')
annotated_categories_df = df[columns_to_extract]

In [578]:
all_rules_categories_df = pd.DataFrame.from_dict(all_rules_categories)

In [621]:
all_rules_categories_df.to_csv('predictions-rules-gpt4_mini.csv')

In [575]:
import re

def extract_binary(value, error_list):
"""
Returns a 2 if no 0 or 1 can be found in the answer string.
"""
    if isinstance(value, int):
        if value in (0, 1):
            return value
        error_list.append(value)
        return 2
    
    if isinstance(value, str):
        match = re.search(r"\b([01])\b", value)
        if match:
            return int(match.group(1))
        else:
            error_list.append(value)
            return 2
    
    error_list.append(value)
    return 2

In [579]:
errors = []

for col in all_rules_categories_df.columns:
    all_rules_categories_df[col] = all_rules_categories_df[col].apply(
        lambda x: extract_binary(x, errors))

In [580]:
errors

['I’m unable to access external links directly. Please provide the text of the rule, and I can help determine if it prohibits harassment.',
 'I’m unable to access external links. Please provide the text of the rule, and I can help determine if it prohibits porn.',
 'I’m unable to access external links directly. Could you please provide the text of the rule or more details? That way, I can help determine if it prohibits violence or incitement.',
 'I’m unable to access external links. Please provide the text of the rule, and I’ll help determine if it prohibits false information, misinformation, disinformation, conspiracy theories, defamation, or propaganda.',
 'I cannot access external links. Please provide the text of the rule so I can determine if it prohibits doxxing.']

In [581]:
all_rules_categories_df = all_rules_categories_df.astype(int)

# comparison of F1 from regex, GPT-4-mini, GPT-5 multiple prompts (each cat), GPT-5 multi single prompt

In [606]:
def label_error_type(y_true, y_pred):
    """
    Gibt TP/TN/FP/FN zurück für binäre Klassifikation.
    1 = positive Klasse, 0 = negative Klasse
    """
    if y_true == 1 and y_pred == 1:
        return "TP"
    elif y_true == 0 and y_pred == 0:
        return "TN"
    elif y_true == 0 and y_pred == 1:
        return "FP"
    elif y_true == 1 and y_pred == 0:
        return "FN"
    else:
        return "Unknown"

all_error_dfs = []

for col in all_rules_categories_df.columns:

    df_temp = pd.DataFrame({
        "category": col,
        "y_true": annotated_categories_df[col],
        "y_pred": all_rules_categories_df[col],
        "translated_rules": annotated_categories_df["translated_rules"]})

    df_temp["error_type"] = df_temp.apply(
        lambda row: label_error_type(row["y_true"], row["y_pred"]), axis=1)

    errors_only = df_temp[df_temp["error_type"].isin(["FP", "FN"])]
    errors_only = errors_only[["category", "y_pred", "y_true", "translated_rules"]]
    all_error_dfs.append(errors_only)

errors_df = pd.concat(all_error_dfs, ignore_index=True)

In [72]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "category": [
        "ads","automated_tools_bots","be_nice","block_report_mute","brigading_dogpiling",
        "codes_conduct","consequences_moderation","copyright_piracy_credit","cw_general",
        "doxxing_personal_info","far_right_extremist_nationalism_fascism","harassment",
        "hate_speech","illegal","impersonation","language_specific","min_age",
        "mis_disinfo_conspiracy_diffamation_propaganda","porn","private_no_reg",
        "spam","topic_specific","violence"
    ],
    "F1_regex": [
        74.8,69.4,65.2,61.5,95.7,68.9,59.2,70.6,90.7,91.1,
        78.1,94.3,86.3,80.6,63.0,40.6,28.1,85.9,57.0,34.1,
        97.1,31.6,78.3
    ],
    "F1_GPT-4-mini_experiment": [
        84.85,68.24,61.80,70.37,63.89,16.71,71.72,76.06,
        93.46,81.63,60.69,59.56,91.98,82.69,84.00,70.83,
        62.92,87.00,68.71,41.67,83.49,11.26,77.15
    ],
    "F1_GPT-5-multi_experiment": [
        68.61,41.38,62.45,69.57,25.34,16.67,79.45,26.67,
        96.99,42.97,51.11,54.03,81.78,88.35,41.82,
        78.57,90.32,55.84,86.36,50.76,64.60,44.21,63.86
    ],
    "F1_GPT-5-single_prompt_experiment": np.nan
})

# -------------------------------------------------------

rename_map = {
    'porn': 'adult_content',
    'porn_nocw': 'adult_content_cw',
    'NSFW': 'nsfw',
    'nsfw_no_cw': 'nsfw_cw',
    'CP': 'CSAM',
    'zoophilia': 'animal_cruelty',
    'doxxing_personal_info': 'privacy',
    'violent_no_cw': 'violence_cw',
    'politics_no_cw': 'politics_cw',
    'ads': 'advertising',
    'mis_disinfo_conspiracy_diffamation_propaganda': 'misinfo',
    'far_right_extremist_nationalism_fascism': 'nazis',
    'ai_content': 'ai_generated',
    'dick': 'dont_be_dick'
}

df["category"] = df["category"].replace(rename_map)

# -------------------------------------------------------

score_map = {
    "CSAM": 89.66,
    "adult_content": 90.32,
    "adult_content_cw": 82.96,
    "advertising": 91.28,
    "ai_generated": 85.71,
    "animal_cruelty": 80.00,
    "automated_tools_bots": 55.07,
    "be_nice": 88.14,
    "block_report_mute": 83.64,
    "codes_conduct": 51.91,
    "consequences_moderation": 75.47,
    "copyright_piracy_credit": 88.64,
    "crypto": 90.48,
    "cw_general": 53.66,
    "dont_be_dick": 53.91,
    "harassment": 78.03,
    "hate_speech": 97.79,
    "illegal": 93.63,
    "impersonation": 85.19,
    "incitement_violence": 85.71,
    "language_specific": 95.45,
    "misinfo": 95.65,
    "nazis": 84.21,
    "nsfw": 88.00,
    "nsfw_cw": 87.76,
    "politics": 44.44,
    "politics_cw": 92.31,
    "privacy": 87.23,
    "private_no_reg": 78.35,
    "religion": 100.00,
    "scrape_model_training": 85.71,
    "spam": 89.29,
    "spoilers": 0.00,
    "violence": 71.03,
    "violence_cw": 94.44
}

# -------------------------------------------------------

missing_categories = sorted(set(score_map.keys()) - set(df["category"]))

missing_df = pd.DataFrame({
    "category": missing_categories,
    "F1_regex": [np.nan] * len(missing_categories),
    "F1_GPT-4-mini_experiment": [np.nan] * len(missing_categories),
    "F1_GPT-5-multi_experiment": [np.nan] * len(missing_categories),
    "F1_GPT-5-single_prompt_experiment": [score_map[c] for c in missing_categories]
})

df = pd.concat([df, missing_df], ignore_index=True)

df["F1_GPT-5-single_prompt_experiment"] = df["category"].map(score_map)

# -------------------------------------------------------

avg_rows = pd.DataFrame([
{
    "category": "micro f1 avg",
    "F1_regex": "",
    "F1_GPT-4-mini_experiment": 85.48,
    "F1_GPT-5-multi_experiment": 80.42,
    "F1_GPT-5-single_prompt_experiment": 95.93
},
{
    "category": "macro f1 avg",
    "F1_regex": "",
    "F1_GPT-4-mini_experiment": 52.48,
    "F1_GPT-5-multi_experiment": 73.25,
    "F1_GPT-5-single_prompt_experiment": 90.77,
}
])

df_markdown = pd.concat([df, avg_rows], ignore_index=True)

# -------------------------------------------------------

def highlight_max(row):
    f1_cols = [
        "F1_regex",
        "F1_GPT-4-mini_experiment",
        "F1_GPT-5-multi_experiment",
        "F1_GPT-5-single_prompt_experiment",
    ]
    numeric_values = pd.to_numeric(row[f1_cols], errors='coerce')
    max_val = numeric_values.max()

    for col in f1_cols:
        val = row[col]
        if pd.to_numeric(val, errors="coerce") == max_val:
            row[col] = f"**{val}**"
        else:
            row[col] = f"{val}"
    return row

df_markdown = df_markdown.apply(highlight_max, axis=1)


print(df_markdown.to_markdown(index=False))


| category                | F1_regex   |   F1_GPT-4-mini_experiment | F1_GPT-5-multi_experiment   | F1_GPT-5-single_prompt_experiment   |
|:------------------------|:-----------|---------------------------:|:----------------------------|:------------------------------------|
| advertising             | 74.8       |                      84.85 | 68.61                       | **91.28**                           |
| automated_tools_bots    | **69.4**   |                      68.24 | 41.38                       | 55.07                               |
| be_nice                 | 65.2       |                      61.8  | 62.45                       | **88.14**                           |
| block_report_mute       | 61.5       |                      70.37 | 69.57                       | **83.64**                           |
| brigading_dogpiling     | **95.7**   |                      63.89 | 25.34                       | nan                                 |
| codes_conduct           | **68.9

| category                | F1_regex   |   F1_GPT-4-mini_experiment | F1_GPT-5-multi_experiment   | F1_GPT-5-single_prompt_experiment   |
|:------------------------|:-----------|---------------------------:|:----------------------------|:------------------------------------|
| advertising             | 74.8       |                      84.85 | 68.61                       | **91.28**                           |
| automated_tools_bots    | **69.4**   |                      68.24 | 41.38                       | 55.07                               |
| be_nice                 | 65.2       |                      61.8  | 62.45                       | **88.14**                           |
| block_report_mute       | 61.5       |                      70.37 | 69.57                       | **83.64**                           |
| brigading_dogpiling     | **95.7**   |                      63.89 | 25.34                       | nan                                 |
| codes_conduct           | **68.9**   |                      16.71 | 16.67                       | 51.91                               |
| consequences_moderation | 59.2       |                      71.72 | **79.45**                   | 75.47                               |
| copyright_piracy_credit | 70.6       |                      76.06 | 26.67                       | **88.64**                           |
| cw_general              | 90.7       |                      93.46 | **96.99**                   | 53.66                               |
| privacy                 | **91.1**   |                      81.63 | 42.97                       | 87.23                               |
| nazis                   | 78.1       |                      60.69 | 51.11                       | **84.21**                           |
| harassment              | **94.3**   |                      59.56 | 54.03                       | 78.03                               |
| hate_speech             | 86.3       |                      91.98 | 81.78                       | **97.79**                           |
| illegal                 | 80.6       |                      82.69 | 88.35                       | **93.63**                           |
| impersonation           | 63.0       |                      84    | 41.82                       | **85.19**                           |
| language_specific       | 40.6       |                      70.83 | 78.57                       | **95.45**                           |
| min_age                 | 28.1       |                      62.92 | **90.32**                   | nan                                 |
| misinfo                 | 85.9       |                      87    | 55.84                       | **95.65**                           |
| adult_content           | 57.0       |                      68.71 | 86.36                       | **90.32**                           |
| private_no_reg          | 34.1       |                      41.67 | 50.76                       | **78.35**                           |
| spam                    | **97.1**   |                      83.49 | 64.6                        | 89.29                               |
| topic_specific          | 31.6       |                      11.26 | **44.21**                   | nan                                 |
| violence                | **78.3**   |                      77.15 | 63.86                       | 71.03                               |
| CSAM                    | nan        |                     nan    | nan                         | **89.66**                           |
| adult_content_cw        | nan        |                     nan    | nan                         | **82.96**                           |
| ai_generated            | nan        |                     nan    | nan                         | **85.71**                           |
| animal_cruelty          | nan        |                     nan    | nan                         | **80.0**                            |
| crypto                  | nan        |                     nan    | nan                         | **90.48**                           |
| dont_be_dick            | nan        |                     nan    | nan                         | **53.91**                           |
| incitement_violence     | nan        |                     nan    | nan                         | **85.71**                           |
| nsfw                    | nan        |                     nan    | nan                         | **88.0**                            |
| nsfw_cw                 | nan        |                     nan    | nan                         | **87.76**                           |
| politics                | nan        |                     nan    | nan                         | **44.44**                           |
| politics_cw             | nan        |                     nan    | nan                         | **92.31**                           |
| religion                | nan        |                     nan    | nan                         | **100.0**                           |
| scrape_model_training   | nan        |                     nan    | nan                         | **85.71**                           |
| spoilers                | nan        |                     nan    | nan                         | **0.0**                             |
| violence_cw             | nan        |                     nan    | nan                         | **94.44**                           |
| micro f1 avg            |            |                      85.48 | 80.42                       | **95.93**                           |
| macro f1 avg            |            |                      52.48 | 73.25                       | **90.77**                           |